# Part 2 — Group Presentation & Notebook

**Group members:** 

- Dargys Perez : `Data Engineer`
  
- Jessika Rosen : `Accounting Manager`
  
- Cecilia Back : `Data Analyst`

**The invoice data requirement**

- invoice_no: invoice unique identifying number(invoice_id/invoice_line)
- invoice_line_no: identifying number for line number.(invoice_line_id/invoice_line)
- invoice_date: date when invoice is issued(invoice_date/invoice_line)
- invoice_type: invoice or credit note (corrective invoice)(invoice_type/invoice_line)
- seller_company_name: TechDist GmbH
- seller_country: country of the seller — always DE for TechDist GmbH
- seller_vat_reg_no: seller's VAT registration number
- seller_iban: Bank account reference
- buyer_company_name: name of the buying company(name/customers)
- buyer_country: country where the buying company is registered(country/customers)
- buyer_vat_reg_no: buyer's VAT registration number(Registration no/customers(here are two options but the VAT no has two missing fields and seems uncompleted information))
- quantity: item quantity invoiced (quantity/order_line? and quantity/invoice_line)
- unit_price: item unit price(unit_price/order_line? and unit_price/invoice_line)
- invoiced_amount_net: net amount invoiced (without VAT)(line_amount/invoice_line)
- invoice_currency: EUR()
- vat_rate: rate applicable for sales item(vat_rate/invoice_line)
- vat_amount: calculated VAT(vat_amount)
- vat_currency: EUR (currency/invoice_line)
- invoiced_amount_total: total amount invoice(total_amount/invoice_line)
- reverse_charge: VAT to be accounted for by the cross-border buyer if buyer has VAT registration(vat_category_code)
- corrective_invoice_ref: for example if invoice is a credit note, it has been issued to correct earlier issued invoice. The original invoice_no is required here (invoice_ref, refunded)
- invoice_status: whether the invoice has been issued or cancelled(invoice_status/ invoice_line)

**The 10-day rule**

From the date the customer order has been delivered, the invoice must be issued within 10 days since the delivery date. (invoice_date(invoice_line) - delivered_date(delivery_confirm_date/deliveries) <= 10)

In [ ]:
# using pandas for data analysis
import pandas as pd

In [13]:
# load the relevant data for the 10 rules analysis
invoice_line = pd.read_csv("invoice_line.csv")
deliveries = pd.read_csv("deliveries.csv")
customers = pd.read_csv("customers.csv")

In [14]:
deliveries.head()

,delivery_confirm_id,order_id,delivery_confirm_date,delivery_country
0,CONF-747653,ORD-2027-296460,2027-02-10,France
1,CONF-650282,ORD-2026-865441,2027-01-13,France
2,CONF-997937,ORD-2027-449701,2027-03-03,Italy
3,CONF-101069,ORD-2027-361984,2027-03-19T11:33:00,Germany
4,CONF-406108,ORD-2027-278275,19/03/2027,Netherlands


In [10]:
deliveries.dtypes

delivery_confirm_id      str
order_id                 str
delivery_confirm_date    str
delivery_country         str
dtype: object

In [11]:
invoice_line.dtypes

invoice_line_id            str
invoice_id                 str
invoice_date               str
invoice_type               str
order_id                   str
product_id                 str
product_description        str
quantity                 int64
uom                        str
unit_price             float64
currency                   str
line_amount            float64
vat_rate               float64
vat_amount             float64
total_amount           float64
invoice_status             str
invoice_ref                str
vat_category_code          str
dtype: object

In [5]:
invoice_line.tail()

,invoice_line_id,invoice_id,invoice_date,invoice_type,order_id,product_id,product_description,quantity,uom,unit_price,currency,line_amount,vat_rate,vat_amount,total_amount,invoice_status,invoice_ref,vat_category_code
98,IL-001,INV-2027-642637,2027-03-10,invoice,ORD-2027-850022,PROD-192837,Droxel 65W GaN Charger,86,pcs,28.4,EUR,2442.4,0.0,0.00,2442.40,issued,NaN,AE
99,IL-003,INV-2027-267919,2027-03-09,invoice,ORD-2027-449701,PROD-461829,Plonex Webcam 1080p Pro,36,pcs,44.0,EUR,1584.0,0.0,0.00,1584.00,issued,NaN,AE
100,IL-001,INV-2027-990042,2027-02-14,invoice,ORD-2026-543670,PROD-192837,Droxel 65W GaN Charger,48,pcs,28.4,EUR,1363.2,19.0,259.01,1622.21,issued,NaN,S
101,IL-001,INV-2027-140410,2027-03-05,invoice,ORD-2027-901318,PROD-639182,Blipcore Wireless Adapter AC1200,42,pcs,22.8,EUR,957.6,0.0,0.00,957.60,issued,NaN,AE
102,IL-001,INV-2027-226551,2027-03-09,invoice,ORD-2027-152306,PROD-738291,Quorvex 2TB NVMe SSD M.2,83,pcs,112.5,EUR,9337.5,0.0,0.00,9337.50,cancelled,NaN,AE


In [25]:
customers.tail()

,Customer ID,Name,Customer type,Registration no,VAT no,City,Country,Registration date,Updated date
10,CUST-599370,Nexivo Solutions SARL,business,FR22445566778,FR22445566778,Paris,France,2014-08-07,2019-09-28
11,CUST-349362,Vektaro Sverige AB,business,SE556771234501,SE556771234501,Stockholm,SE,2022-07-25,2025-02-05
12,CUST-365005,Blipware Systems GmbH,business,DE445523901,DE 44552390,Munich,Deutschland,2016-04-12,2016-09-05
13,CUST-302511,Fomtec Vertriebs GmbH,business,DE889967345,NaN,Dusseldorf,DE,2017-06-08,2021-02-13
14,CUST-741029,Plonix Distributie BV,business,NL553344556B01,NaN,Utrecht,NLD,2011-10-26,2016-05-21


Address at least one concrete data quality issue found in the datasets

In [15]:
#Join datasets
df = invoice_line.merge(
    deliveries,
    on="order_id",
    how="left"
)

In [16]:
#Convert to datetime
df["invoice_date"] = pd.to_datetime(df["invoice_date"])


In [17]:
# the delivery_confirm_date has data quality issues, the date is given in differents formats and I need first to normalize.
df["delivery_confirm_date"] = pd.to_datetime(
    df["delivery_confirm_date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)


In [18]:
# I wanted to check if the datatypes where compatible with each other.
df.dtypes

invoice_line_id                     str
invoice_id                          str
invoice_date             datetime64[us]
invoice_type                        str
order_id                            str
product_id                          str
product_description                 str
quantity                          int64
uom                                 str
unit_price                      float64
currency                            str
line_amount                     float64
vat_rate                        float64
vat_amount                      float64
total_amount                    float64
invoice_status                      str
invoice_ref                         str
vat_category_code                   str
delivery_confirm_id                 str
delivery_confirm_date    datetime64[us]
delivery_country                    str
dtype: object

>Note: As we see the invoice_date and  delivery_confirm_date are both in the same format, ready for the next step

In [22]:

df["days_diff"] = (df["invoice_date"] - df["delivery_confirm_date"]).dt.days

In [26]:
df["days_diff"]

0       8
1       5
2       6
3      13
4      20
       ..
98     22
99      6
100    16
101     5
102     6
Name: days_diff, Length: 103, dtype: int64

In [20]:
# Here are we "Grouping By: count()" as we know from SQL, it counts the invoices that where send late and the ones that where on time
df["vida_10_day_rule"] = df["days_diff"].apply(
    lambda x: "Compliant" if x <= 10 else "Late"
)

In [21]:
#Just showing the results
df["vida_10_day_rule"].value_counts()

vida_10_day_rule
Compliant    71
Late         32
Name: count, dtype: int64